In [ ]:
# ======================================================
# Cell 1: Install Dependencies
# ======================================================
import subprocess, sys

for _pkg in ["pandas", "numpy", "matplotlib", "seaborn", "scipy", "openpyxl"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"])

print("All dependencies installed.")

In [ ]:
# ======================================================
# Cell 2: Platform Detection & Path Configuration
# ======================================================
import os, sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT_DIR = "/content/drive/MyDrive"
    print("Running on Google Colab - Google Drive mounted.")
else:
    ROOT_DIR = os.path.dirname(os.getcwd())
    print(f"Running locally - root: {ROOT_DIR}")

RESULTS_DIR = os.path.join(ROOT_DIR, "RESULTS")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results folder: {RESULTS_DIR}")

def pick_file(title="Select a file", filetypes=(("Excel files", "*.xlsx"),)):
    """Returns a file path - uses Colab uploader online, prompts for path offline."""
    if IS_COLAB:
        from google.colab import files as _colab_files
        uploaded = _colab_files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        fname = next(iter(uploaded))
        path = f"/tmp/{fname}"
        with open(path, "wb") as fh:
            fh.write(uploaded[fname])
        return path
    else:
        print(f"\n{title}")
        print("   Paste the full path to your file and press Enter.")
        print("   Example: /Users/you/project/Datasets/mapped-dataset-36-20.xlsx")
        path = input("   Path: ").strip().strip('"').strip("'")
        if not path:
            raise RuntimeError("No path entered.")
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"File not found at:\n  {path}\nCheck the path and re-run this cell."
            )
        return path

print("Platform setup complete.")

# RTW Scheduling Algorithm Analysis

Reusable analysis pipeline for the **Results_Dx.xlsx** family of datasets.

For each input dataset this notebook produces:

1. A **per-dataset analysis workbook** (`Results_<DATASET>_Analysis.xlsx`) with five metric sheets that reference the source data via Excel formulas — so values stay in sync if you tweak `Summary`/`Wait_Req_*` rows.
2. A row in a **consolidated results workbook** (`Consolidated_Results.xlsx`) so you can compare metrics across datasets.

To analyze another dataset, change `INPUT_FILE` and `DATASET_NAME` in the Configuration cell and run all cells again. Re-running with the same `DATASET_NAME` is **idempotent** — that dataset's rows are replaced, not duplicated.

## Metrics computed
- **1. Avg BV across RTWs** — per algorithm, average BV over all RTW fractions
- **2. Avg Coverage across RTWs** — per algorithm, average coverage over all RTW fractions
- **3. BV loss vs MemLess (%)** — per algorithm per RTW, plus row average
- **4. Coverage gap vs MemLess (pp)** — per algorithm per RTW, plus row average
- **5. Avg skipped RTWs per RTW fraction** — per algorithm per RTW (averaged across requirements)

## Assumed input structure
The notebook auto-detects algorithm list, RTW fractions, and requirement count, but it expects:
- A `Summary` sheet with columns `Algorithm`, `RTW Fraction`, `BV`, `Coverage`. Rows interleaved by RTW: for each RTW fraction the algorithms appear in a fixed order.
- One `Wait_Req_<Algorithm>` sheet per algorithm, with `Req_ID` in column A and one column per RTW fraction (rows = requirements).


## 1. Configuration
Edit these values, then run all cells.

In [13]:
import os
from pathlib import Path

INPUT_FILE         = pick_file("Select Results_Dx.xlsx file")
DATASET_NAME       = "D3"                          # short id used in the consolidated file
PER_DATASET_OUTPUT = os.path.join(RESULTS_DIR, "Results_Analysis.xlsx")  # per-dataset output
# CONSOLIDATED_FILE  = "Consolidated_Results.xlsx"   # cross-dataset rollup (appended/updated)

# Baseline algorithm for loss/gap metrics (metrics 3 and 4)
BASELINE = "MemLess"

# Optional: override detected algorithm order. Leave as None to auto-detect from Summary.
ALGO_ORDER_OVERRIDE = None    # e.g. ["MemLess", "Aging", "DRR", "RR", "WRR"]

## 2. Imports

In [14]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

assert Path(INPUT_FILE).exists(), f"Input file not found: {INPUT_FILE}"
print(f"Input: {INPUT_FILE}")
print(f"Dataset name: {DATASET_NAME}")


Input: Results_D3.xlsx
Dataset name: D3


## 3. Inspect input and detect structure
Detect algorithms, RTW fractions, and the number of requirements automatically. Validates that the Summary sheet has the expected columns and interleaved-by-RTW layout.


In [15]:
summary = pd.read_excel(INPUT_FILE, sheet_name="Summary")
required = {"Algorithm", "RTW Fraction", "BV", "Coverage"}
assert required.issubset(summary.columns), \
    f"Summary sheet must contain {required}; got {set(summary.columns)}"

if ALGO_ORDER_OVERRIDE is not None:
    ALGOS = ALGO_ORDER_OVERRIDE
else:
    ALGOS = list(dict.fromkeys(summary["Algorithm"]))

RTW_FRACS = sorted(summary["RTW Fraction"].unique())
N_RTW = len(RTW_FRACS)

print(f"Algorithms ({len(ALGOS)}): {ALGOS}")
print(f"RTW fractions ({N_RTW}): {RTW_FRACS}")
assert BASELINE in ALGOS, f"Baseline {BASELINE!r} not in {ALGOS}"

wait_baseline = pd.read_excel(INPUT_FILE, sheet_name=f"Wait_Req_{BASELINE}")
N_REQS = len(wait_baseline)
print(f"Requirements per Wait_Req sheet: {N_REQS}")


Algorithms (5): ['MemLess', 'Aging', 'DRR', 'RR', 'WRR']
RTW fractions (8): [np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8)]
Requirements per Wait_Req sheet: 36


## 4. Helper functions and styling

In [16]:
def summary_row(algo, rtw_idx):
    """Excel row in Summary for (algo, rtw_idx). Assumes interleaved-by-RTW layout:
    row = 2 + rtw_idx * len(ALGOS) + ALGOS.index(algo)."""
    return 2 + rtw_idx * len(ALGOS) + ALGOS.index(algo)

# Sanity-check the assumed layout
_r = summary_row(ALGOS[0], 0)
assert summary.iloc[_r - 2]["Algorithm"] == ALGOS[0], (
    "Summary sheet does not match the expected interleaved-by-RTW layout. "
    "Set ALGO_ORDER_OVERRIDE to fix the algorithm order."
)

HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=11)
HEADER_FILL = PatternFill("solid", start_color="305496")
TITLE_FONT  = Font(name="Arial", bold=True, size=14, color="1F4E79")
NOTE_FONT   = Font(name="Arial", italic=True, size=10, color="595959")
DEFAULT_FONT = Font(name="Arial", size=11)
HIGHLIGHT_FILL = PatternFill("solid", start_color="FFF2CC")
_thin = Side(border_style="thin", color="BFBFBF")
BORDER = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
CENTER = Alignment(horizontal="center", vertical="center")
LEFT   = Alignment(horizontal="left",   vertical="center")

def style_header(c):
    c.font = HEADER_FONT; c.fill = HEADER_FILL
    c.alignment = CENTER; c.border = BORDER

def style_data(c, num_format=None, highlight=False):
    c.font = DEFAULT_FONT; c.alignment = CENTER; c.border = BORDER
    if num_format: c.number_format = num_format
    if highlight:  c.fill = HIGHLIGHT_FILL

def write_title(ws, text, ncols, row=1):
    ws.cell(row=row, column=1, value=text).font = TITLE_FONT
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    ws.cell(row=row, column=1).alignment = LEFT

def write_note(ws, text, ncols, row):
    ws.cell(row=row, column=1, value=text).font = NOTE_FONT
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    ws.cell(row=row, column=1).alignment = LEFT

def set_widths(ws, widths):
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w


## 5. Build the per-dataset analysis workbook

Adds five metric sheets to the source workbook (formulas reference `Summary` and `Wait_Req_<Algo>`, so values stay in sync if the source changes). Saved as `PER_DATASET_OUTPUT`.


In [17]:
wb = load_workbook(INPUT_FILE)

# Wipe any prior analysis sheets so re-running is idempotent
for name in ["Avg_BV", "Avg_Coverage", "BV_Loss_vs_MemLess",
             "Coverage_Gap_vs_MemLess", "Avg_Skipped_per_RTW"]:
    if name in wb.sheetnames:
        del wb[name]

WAIT_COLS = [get_column_letter(2 + k) for k in range(N_RTW)]
WAIT_LAST = N_REQS + 1
non_base = [a for a in ALGOS if a != BASELINE]
n_cols_matrix = 1 + N_RTW + 1

# --- 1_Avg_BV ---
ws = wb.create_sheet("Avg_BV")
# write_title(ws, "Metric 1 — Average BV across RTW fractions", 3)
write_note(ws, f"AvgBV = average of BV over the {N_RTW} RTW fractions. Source: Summary!C. Denominator = {N_RTW}.", 3, row=2)
for j, h in enumerate(["Algorithm", "Avg BV", "# RTW fractions"], start=1):
    style_header(ws.cell(row=4, column=j, value=h))
for i, algo in enumerate(ALGOS):
    r = 5 + i
    ws.cell(row=r, column=1, value=algo); style_data(ws.cell(row=r, column=1))
    refs = ",".join(f"Summary!C{summary_row(algo, k)}" for k in range(N_RTW))
    ws.cell(row=r, column=2, value=f"=AVERAGE({refs})")
    style_data(ws.cell(row=r, column=2), num_format="0.00", highlight=(algo == BASELINE))
    ws.cell(row=r, column=3, value=N_RTW); style_data(ws.cell(row=r, column=3))
set_widths(ws, [18, 16, 18])

# --- 2_Avg_Coverage ---
ws = wb.create_sheet("Avg_Coverage")
# write_title(ws, "Metric 2 — Average Coverage (%) across RTW fractions", 3)
write_note(ws, f"AvgCoverage = average of Coverage over the {N_RTW} RTW fractions. Source: Summary!D. Denominator = {N_RTW}.", 3, row=2)
for j, h in enumerate(["Algorithm", "Avg Coverage (%)", "# RTW fractions"], start=1):
    style_header(ws.cell(row=4, column=j, value=h))
for i, algo in enumerate(ALGOS):
    r = 5 + i
    ws.cell(row=r, column=1, value=algo); style_data(ws.cell(row=r, column=1))
    refs = ",".join(f"Summary!D{summary_row(algo, k)}" for k in range(N_RTW))
    ws.cell(row=r, column=2, value=f"=AVERAGE({refs})")
    style_data(ws.cell(row=r, column=2), num_format="0.00", highlight=(algo == BASELINE))
    ws.cell(row=r, column=3, value=N_RTW); style_data(ws.cell(row=r, column=3))
set_widths(ws, [18, 22, 18])

# --- 3_BV_Loss_vs_<BASELINE> ---
ws = wb.create_sheet("BV_Loss_vs_MemLess")
# write_title(ws, f"Metric 3 — BV loss (%) vs {BASELINE}", n_cols_matrix)
write_note(ws, f"BVLoss(alg, rtw) = (BV_{BASELINE} − BV_alg)/BV_{BASELINE} × 100. Last column = average across the {N_RTW} RTWs.", n_cols_matrix, row=2)
style_header(ws.cell(row=4, column=1, value="Algorithm"))
for k, rtw in enumerate(RTW_FRACS):
    style_header(ws.cell(row=4, column=2 + k, value=f"RTW={rtw}"))
avg_col = 2 + N_RTW
style_header(ws.cell(row=4, column=avg_col, value="Avg BV Loss (%)"))
for i, algo in enumerate(non_base):
    r = 5 + i
    ws.cell(row=r, column=1, value=algo); style_data(ws.cell(row=r, column=1))
    for k in range(N_RTW):
        b = f"Summary!C{summary_row(BASELINE, k)}"
        a = f"Summary!C{summary_row(algo, k)}"
        ws.cell(row=r, column=2 + k, value=f"=IF({b}=0,0,({b}-{a})/{b}*100)")
        style_data(ws.cell(row=r, column=2 + k), num_format="0.00")
    start, end = get_column_letter(2), get_column_letter(1 + N_RTW)
    ws.cell(row=r, column=avg_col, value=f"=AVERAGE({start}{r}:{end}{r})")
    style_data(ws.cell(row=r, column=avg_col), num_format="0.00", highlight=True)
set_widths(ws, [14] + [12]*N_RTW + [18])

# --- 4_Coverage_Gap_vs_<BASELINE> ---
ws = wb.create_sheet("Coverage_Gap_vs_MemLess")
# write_title(ws, f"Metric 4 — Coverage gap (pp) vs {BASELINE}", n_cols_matrix)
write_note(ws, f"CoverageGap(alg, rtw) = Coverage_{BASELINE} − Coverage_alg (pp). Last column = average across the {N_RTW} RTWs.", n_cols_matrix, row=2)
style_header(ws.cell(row=4, column=1, value="Algorithm"))
for k, rtw in enumerate(RTW_FRACS):
    style_header(ws.cell(row=4, column=2 + k, value=f"RTW={rtw}"))
style_header(ws.cell(row=4, column=avg_col, value="Avg Coverage Gap (pp)"))
for i, algo in enumerate(non_base):
    r = 5 + i
    ws.cell(row=r, column=1, value=algo); style_data(ws.cell(row=r, column=1))
    for k in range(N_RTW):
        b = f"Summary!D{summary_row(BASELINE, k)}"
        a = f"Summary!D{summary_row(algo, k)}"
        ws.cell(row=r, column=2 + k, value=f"={b}-{a}")
        style_data(ws.cell(row=r, column=2 + k), num_format="0.00")
    start, end = get_column_letter(2), get_column_letter(1 + N_RTW)
    ws.cell(row=r, column=avg_col, value=f"=AVERAGE({start}{r}:{end}{r})")
    style_data(ws.cell(row=r, column=avg_col), num_format="0.00", highlight=True)
set_widths(ws, [14] + [12]*N_RTW + [22])

# --- 5B_Avg_Skipped_per_RTW ---
ws = wb.create_sheet("Avg_Skipped_per_RTW")
# write_title(ws, f"Metric 5B — Average skipped RTWs per RTW fraction (across {N_REQS} requirements)", n_cols_matrix)
write_note(ws, f"AvgSkipped(A, r) = AVERAGE of Wait_Req_A column for r over rows 2..{WAIT_LAST}. Denominator = {N_REQS}.", n_cols_matrix, row=2)
style_header(ws.cell(row=4, column=1, value="Algorithm"))
for k, rtw in enumerate(RTW_FRACS):
    style_header(ws.cell(row=4, column=2 + k, value=f"RTW={rtw}"))
style_header(ws.cell(row=4, column=avg_col, value="Row Avg across RTWs"))
for i, algo in enumerate(ALGOS):
    r = 5 + i
    ws.cell(row=r, column=1, value=algo); style_data(ws.cell(row=r, column=1))
    sn = f"Wait_Req_{algo}"
    for k, col in enumerate(WAIT_COLS):
        ws.cell(row=r, column=2 + k, value=f"=AVERAGE('{sn}'!{col}2:{col}{WAIT_LAST})")
        style_data(ws.cell(row=r, column=2 + k), num_format="0.0000")
    start, end = get_column_letter(2), get_column_letter(1 + N_RTW)
    ws.cell(row=r, column=avg_col, value=f"=AVERAGE({start}{r}:{end}{r})")
    style_data(ws.cell(row=r, column=avg_col), num_format="0.0000", highlight=True)
set_widths(ws, [14] + [12]*N_RTW + [22])

# Put analysis sheets right after Summary
desired = ["Summary", "Visualizations",
           "Avg_BV", "Avg_Coverage",
           "BV_Loss_vs_MemLess", "Coverage_Gap_vs_MemLess",
           "Avg_Skipped_per_RTW"]
remaining = [s for s in wb.sheetnames if s not in desired]
wb._sheets = [wb[n] for n in desired + remaining if n in wb.sheetnames]

wb.save(PER_DATASET_OUTPUT)
print(f"Saved per-dataset analysis → {PER_DATASET_OUTPUT}")


Saved per-dataset analysis → Results_D3_Analysis.xlsx


## 6. Compute metric values in pandas (for the consolidated rollup)

We recompute the same metrics in pandas so the consolidated workbook contains plain values — that way each row stands on its own, with no cross-workbook formula references.


In [18]:
# Metrics 1 & 2
m1 = summary.groupby("Algorithm")["BV"].mean().reindex(ALGOS).rename("Avg_BV")
m2 = summary.groupby("Algorithm")["Coverage"].mean().reindex(ALGOS).rename("Avg_Coverage")

# Metric 3: BV loss vs baseline at each RTW + row avg
piv_bv = summary.pivot(index="Algorithm", columns="RTW Fraction", values="BV").reindex(ALGOS)
base_bv = piv_bv.loc[BASELINE]
bv_loss = piv_bv.apply(lambda row: (base_bv - row) / base_bv * 100, axis=1).drop(index=BASELINE)
bv_loss["Avg_BV_Loss_%"] = bv_loss.iloc[:, :N_RTW].mean(axis=1)

# Metric 4: coverage gap vs baseline at each RTW + row avg
piv_cov = summary.pivot(index="Algorithm", columns="RTW Fraction", values="Coverage").reindex(ALGOS)
base_cov = piv_cov.loc[BASELINE]
cov_gap = piv_cov.apply(lambda row: base_cov - row, axis=1).drop(index=BASELINE)
cov_gap["Avg_Coverage_Gap_pp"] = cov_gap.iloc[:, :N_RTW].mean(axis=1)

# Metric 5B: avg skipped per RTW, per algorithm
rows_5b = {}
for algo in ALGOS:
    df = pd.read_excel(INPUT_FILE, sheet_name=f"Wait_Req_{algo}")
    means = df.iloc[:, 1:1 + N_RTW].mean()
    means.index = RTW_FRACS
    rows_5b[algo] = means
m5b = pd.DataFrame(rows_5b).T.reindex(ALGOS)
m5b["Row_Avg"] = m5b.iloc[:, :N_RTW].mean(axis=1)

print("Metric 1 — Avg BV:")
print(m1.round(2).to_string())
print("\nMetric 2 — Avg Coverage:")
print(m2.round(2).to_string())
print(f"\nMetric 3 — BV loss vs {BASELINE} (%):")
print(bv_loss.round(2).to_string())
print(f"\nMetric 4 — Coverage gap vs {BASELINE} (pp):")
print(cov_gap.round(2).to_string())
print("\nMetric 5 — Avg skipped per RTW:")
print(m5b.round(4).to_string())


Metric 1 — Avg BV:
Algorithm
MemLess    226.0
Aging      104.0
DRR        226.0
RR         255.5
WRR        255.5

Metric 2 — Avg Coverage:
Algorithm
MemLess    29.38
Aging      14.38
DRR        29.38
RR         59.38
WRR        59.38

Metric 3 — BV loss vs MemLess (%):
RTW Fraction  0.1     0.2   0.3     0.4    0.5     0.6     0.7    0.8  Avg_BV_Loss_%
Algorithm                                                                           
Aging         0.0  100.00  0.00  100.00   0.00  100.00    0.00  100.0          50.00
DRR           0.0    0.00  0.00    0.00   0.00    0.00    0.00    0.0           0.00
RR           -8.0   -3.28 -3.28   -3.28  36.48  -22.54 -126.64   29.1         -12.68
WRR          -8.0   -3.28 -3.28   -3.28  36.48  -22.54 -126.64   29.1         -12.68

Metric 4 — Coverage gap vs MemLess (pp):
RTW Fraction  0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  Avg_Coverage_Gap_pp
Algorithm                                                                
Aging           0   30    0  